# 调频

Canonical workflow notebook for frequency-regulation aging studies. 所有用户可变参数集中在下一格 `CONFIG`。


## 1. 用户配置


In [ ]:
# 所有工况变体只修改本单元
CONFIG = {
    "cell": "587Ah",
    "output_name": "调频",
    "run_mode": "smoke",  # "smoke" / "study"
    "current_a": 293.5,
    "nominal_capacity_ah": 587.0,
    "temperature_c": 25.0,
    "initial_soc": 0.60,
    "use_equivalent_frequency": True,
    "datasets": [
        # {"cell": "587Ah", "temperature_c": 25, "test_type": "调频", "kind": "processed", "require_unique": False},
    ],
    "modes": {
        "smoke": {"real_days_total": 1, "day_acceleration_factor": 1, "rpt_every_real_days": 1, "showprogress": False, "parallel": False, "return_solutions": True},
        "study": {"real_days_total": 4000, "day_acceleration_factor": 50, "rpt_every_real_days": 50, "showprogress": True, "parallel": True, "max_workers": 2, "return_solutions": False},
    },
    "var_pts": {"x_n": 5, "x_s": 5, "x_p": 5, "r_n": 20, "r_p": 20},
    "scenarios": [
        {"name": "pulse_10s_1efc", "display_name": "10 s 脉冲 / 1 EFC per day", "pulse_seconds": 10, "total_pulses_per_day": 1440, "sample_period_seconds": 2, "enabled": True},
        {"name": "pulse_10s_2efc", "display_name": "10 s 脉冲 / 2 EFC per day", "pulse_seconds": 10, "total_pulses_per_day": 2880, "sample_period_seconds": 2, "enabled": True},
    ],
}
ANALYSIS = {"waveform": True, "aging_plots": True, "export": True}


## 2. 环境与导入


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
plt.style.use("science")
plt.rcParams["font.family"] = "Calibri, Microsoft YaHei"
import pandas as pd

SEARCH_ROOT = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in (SEARCH_ROOT, *SEARCH_ROOT.parents):
    if (candidate / "src" / "easy_imports.py").exists() and (candidate / "pyproject.toml").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"Cannot locate BatteryProject root from {SEARCH_ROOT}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.easy_imports import configure_notebook_environment
from src.workflows.frequency import FrequencyWorkflowSpec, build_frequency_waveform_preview, prepare_frequency_workflow, run_frequency_workflow

PROJECT_ROOT, WORKSPACE_ROOT, PARAMS_ROOT = configure_notebook_environment(project_root=PROJECT_ROOT, include_workspace_root=True)


## 3. 配置校验与数据查询


In [ ]:
spec = FrequencyWorkflowSpec.from_mapping(CONFIG)
prepared = prepare_frequency_workflow(spec)
dataset_entries = []
for query in spec.datasets:
    dataset_entries.extend(query.resolve(WORKSPACE_ROOT))
print(f"run_mode={spec.run_mode}; scenarios={len(prepared['scenarios'])}; datasets={len(dataset_entries)}")
prepared["scenario_table"]


## 4. 等效波形检查


In [ ]:
if ANALYSIS["waveform"]:
    preview = build_frequency_waveform_preview(spec)
    comparison_df = preview["comparison_df"]
    raw_time_s, raw_current_a = preview["raw_time_s"], preview["raw_current_a"]
    equivalent_time_s, equivalent_current_a = preview["equivalent_time_s"], preview["equivalent_current_a"]
    raw_zoom_time_s, raw_zoom_current_a = preview["raw_zoom"]
    equivalent_zoom_time_s, equivalent_zoom_current_a = preview["equivalent_zoom"]
    focus_label = f"{float(preview['focus_scenario']['pulse_seconds']):g} s 脉冲"
    fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
    axes[0, 0].plot(raw_time_s / 3600.0, raw_current_a, linewidth=1.0, color="#1f77b4")
    axes[0, 0].set_title(f"{focus_label}: 原始全波形")
    axes[0, 0].set_xlabel("Waveform time [h]")
    axes[0, 0].set_ylabel("Current [A]")
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 1].plot(equivalent_time_s / 3600.0, equivalent_current_a, linewidth=1.2, color="#d62728")
    axes[0, 1].set_title("等效压缩后: 全波形")
    axes[0, 1].set_xlabel("Waveform time [h]")
    axes[0, 1].set_ylabel("Current [A]")
    axes[0, 1].grid(True, alpha=0.3)
    axes[1, 0].plot(raw_zoom_time_s, raw_zoom_current_a, linewidth=1.0, color="#1f77b4")
    axes[1, 0].set_title("原始波形局部")
    axes[1, 0].set_xlabel("Waveform time [s]")
    axes[1, 0].set_ylabel("Current [A]")
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 1].plot(equivalent_zoom_time_s, equivalent_zoom_current_a, linewidth=1.2, color="#d62728")
    axes[1, 1].set_title("等效压缩后局部")
    axes[1, 1].set_xlabel("Waveform time [s]")
    axes[1, 1].set_ylabel("Current [A]")
    axes[1, 1].grid(True, alpha=0.3)
    comparison_df


## 5. 调频寿命仿真


In [ ]:
result = run_frequency_workflow(spec, project_root=PROJECT_ROOT, workspace_root=WORKSPACE_ROOT)
results = result["results"]
summary_df = result["summary_df"]
print("输出目录:", result["context"].run_dir)
summary_df[[col for col in ["scenario", "real_days_covered", "efc_per_day", "use_equivalent_frequency", "original_pulses_per_day", "simulated_segments_per_day", "pulse_compression_ratio", "simulated_segment_seconds", "final_capacity_retention_pct", "final_rpt_efficiency_pct", "final_q_sei_ah", "final_q_plating_ah"] if col in summary_df.columns]]


## 6. 结果图


In [ ]:
if ANALYSIS["aging_plots"]:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
    for bundle in results.values():
        label = bundle["scenario"]["display_name"]
        main_df = bundle["main_df"]
        rpt_df = bundle["rpt_df"]
        if not main_df.empty:
            axes[0, 0].plot(main_df["real_day"], main_df["q_sei_ah"], marker="o", label=label)
            axes[0, 1].plot(main_df["real_day"], main_df["q_plating_ah"], marker="o", label=label)
        if not rpt_df.empty:
            axes[1, 0].plot(rpt_df["real_day"], rpt_df["capacity_retention"] * 100.0, marker="o", label=label)
            axes[1, 1].plot(rpt_df["real_day"], rpt_df["rpt_efficiency_pct"], marker="o", label=label)
    axes[0, 0].set_title("SEI growth")
    axes[0, 0].set_xlabel("Real day")
    axes[0, 0].set_ylabel("Q_SEI [Ah]")
    axes[0, 1].set_title("Lithium plating")
    axes[0, 1].set_xlabel("Real day")
    axes[0, 1].set_ylabel("Q_plating [Ah]")
    axes[1, 0].set_title("Capacity retention")
    axes[1, 0].set_xlabel("Real day")
    axes[1, 0].set_ylabel("Retention [%]")
    axes[1, 1].set_title("RPT efficiency")
    axes[1, 1].set_xlabel("Real day")
    axes[1, 1].set_ylabel("Efficiency [%]")
    for ax in axes.ravel():
        ax.grid(True, alpha=0.3)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2)
    plot_path = result["context"].plots_dir / "frequency_results.png"
    fig.savefig(plot_path, dpi=180)
    print("结果图:", plot_path)


## 7. Artifact 链接


In [ ]:
pd.DataFrame([{"artifact": key, "path": str(path)} for key, path in result["artifact_paths"].items()])
